In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

df_trips = pd.read_csv('/content/drive/MyDrive/trip.csv')
df_stations = pd.read_csv('/content/drive/MyDrive/station.csv')

In [3]:
df_trips['start_time'] = pd.to_datetime(df_trips['start_time'])
df_trips['end_time'] = pd.to_datetime(df_trips['end_time'])
df_stations['station_name'] = df_stations['station_name'].apply(lambda x: x.strip())

In [4]:
# Question A: How many trips were recorded in total?
len(df_trips)

1569966

In [5]:
# Question B: How many municipalities participated in the bike share program?
len(set(df_stations['municipality']))

4

In [6]:
# Question C: List the number of stations in each municipality.
df_station_count = df_stations.groupby('municipality', as_index = False)['station_name'].count().rename(columns={'station_name':'station_count'})
df_station_count.sort_values(by=['station_count','municipality'],ascending=[False,True])

,municipality,station_count
0,Boston,97
2,Cambridge,28
3,Somerville,12
1,Brookline,5


In [8]:
# Question D: What percentage of registered female users participated in the bike sharing program?
df_trips_gender = df_trips.groupby('gender', as_index = False)['id'].count().rename(columns={'id':'count'});
df_trips_gender['count'] = df_trips_gender['count'] * 100 / df_trips_gender['count'].sum()
female_percentage = df_trips_gender.iloc[0, 1]
round(float(female_percentage), 2)

24.55

In [9]:
# Question E: Find the percentage of trips that were below 60 seconds in duration?
df_trips_duration = df_trips[df_trips['duration'] < 60]
duration_percentage = len(df_trips_duration) * 100 / len(df_trips)
round(duration_percentage, 2)

0.5

In [10]:
# Question F: Find the longest trip recorded along with the corresponding station names.
df_trips_stations = pd.merge(df_trips, df_stations,left_on='start_station_id',right_on='station_id')
df_trips_stations = pd.merge(df_trips_stations, df_stations,left_on='end_station_id',right_on='station_id')

df_trips_stations = df_trips_stations.rename(columns={'station_name_x':'start_station_name','station_name_y':'end_station_name'})
df_trips_longest = df_trips_stations[df_trips_stations['duration'] == df_trips_stations['duration'].max()]
df_trips_longest[['id','start_station_name','end_station_name','duration','bike_id']]

,id,start_station_name,end_station_name,duration,bike_id
534848,541247,Tremont St / West St,Tremont St / West St,9999,T01078


In [11]:
# Question G: What is the ratio of average trip duration for casual customers vs registered customers?
duration_average = df_trips.groupby('user_type', as_index = False)['duration'].mean()
duration_ratio = duration_average.iloc[0, 1] / duration_average.iloc[1, 1]
round(float(duration_ratio), 2)

2.31

In [13]:
# Question H: Find the percentage of roundtrips. A trip is classified as a round-trip if its starting station is the same as its ending station.
roundtrips = len(df_trips[df_trips['start_station_id'] == df_trips['end_station_id']])
roudtrips_percentage = roundtrips * 100 / len(df_trips)
round(roudtrips_percentage, 2)

4.65

In [14]:
# Question I: Find the bikes have travelled to EVERY station in Somerville.
df_trips_stations = df_trips_stations.rename(columns={'municipality_x':'start_municipality','municipality_y':'end_municipality'})
somerville_stations = set(df_stations[df_stations['municipality'] == 'Somerville']['station_id'])
df_trips_somerville = df_trips_stations[(df_trips_stations['start_municipality'] == 'Somerville') | (df_trips_stations['end_municipality'] ==
                                                                                                     'Somerville')]

rows = []

for bike_id, trip_info in df_trips_somerville.groupby('bike_id'):
  visited_stations = list(trip_info['start_station_id']) + list(trip_info['end_station_id'])
  visited_stations = set(visited_stations)

  if somerville_stations.issubset(visited_stations):
    rows.append(bike_id)

df_trips_somerville = pd.DataFrame(rows, columns=['bike_id']).sort_values(by='bike_id')
df_trips_somerville

,bike_id
0,B00001
1,B00014
2,B00018
3,B00028
4,B00092
5,B00172
6,B00215
7,B00217
8,B00242
9,B00314


In [17]:
# Question J: Find the top 10 bikes which have been to the most number of distinct stations.
rows = []

for bike_id, trip_info in df_trips.groupby('bike_id'):
  stations = list(trip_info['start_station_id']) + list(trip_info['end_station_id'])
  distinct_stations_count = len(set(stations))
  rows.append([bike_id, distinct_stations_count])

df_distinct_stations = pd.DataFrame(rows, columns=['bike_id', 'distinct_stations_count']).sort_values(by=['distinct_stations_count','bike_id'],
                                                                                                      ascending=[False,True])
df_distinct_stations.head(10)

,bike_id,distinct_stations_count
17,B00001,134
469,B00456,132
565,B00553,132
926,T01214,132
366,B00353,131
283,B00270,130
331,B00318,130
384,B00371,130
451,B00438,130
495,B00482,130


In [19]:
# Question K: Find the percentage of trips for each municipality.
rows = []
percentage_sum = 0

for m in set(df_stations['municipality']):
  trips_count = len(df_trips_stations[
      (df_trips_stations['start_municipality'] == m) | (df_trips_stations['end_municipality'] == m)
  ])

  trips_percentage = trips_count * 100 / len(df_trips)
  percentage_sum += trips_percentage
  rows.append([round(trips_percentage,2), m])

df_trips_percentage = pd.DataFrame(rows, columns=['trips_percentage', 'municipality']).sort_values(by=['trips_percentage','municipality'],
                                                                                                   ascending=[False,True])
for i in range(len(set(df_stations['municipality']))):
  df_trips_percentage.iloc[i, 0] = df_trips_percentage.iloc[i, 0] * 100 / percentage_sum
  df_trips_percentage.iloc[i, 0] = round(df_trips_percentage.iloc[i, 0], 2)

df_trips_percentage

,trips_percentage,municipality
3,71.51,Boston
0,22.91,Cambridge
2,3.55,Somerville
1,2.03,Brookline
